# Dataset-wide structural validation

Check whether the structure seen in the first notebook holds across the released GBE data, and keep track of anything that needs a closer look. No exclusions or corrections are applied here.

## 1. Load the data

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

# Notebook is in notebooks/; data lives outside the code repository.
PROJECT_DIR = Path.cwd().parents[1]
DATA_PATH = PROJECT_DIR / "data" / "raw" / "gbe" / "Rutledge_GBE_risk_data_TOD.mat"

print("Data file found:", DATA_PATH.is_file())

Data file found: True


In [3]:
mat = loadmat(DATA_PATH, simplify_cells=True)
participants = mat["subjData"]

In [5]:
def get_plays(participant):
    # return a list of trial matrices, even for someone with one play
    if participant["nPlays"] == 1:
        return [participant["data"]]
    return list(participant["data"])

## 2. Check data structure

Check that each participant’s reported number of available plays (`nPlays`) matches the number of trial matrices returned by `get_plays()`.

`get_plays()` uses `nPlays` to interpret the container, so this is a count check rather than an independent check of the container type. Matrix shapes are checked next.

In [7]:
play_count_issues = []
total_plays = 0

for participant in participants:
    plays = get_plays(participant)
    total_plays += len(plays)

    if len(plays) != participant["nPlays"]:
        play_count_issues.append({
            "id": participant["id"],
            "nPlays": participant["nPlays"],
            "available_matrices": len(plays),
        })

print("People checked:", len(participants))
print("Available plays:", total_plays)
print("Play-count mismatches:", len(play_count_issues))

People checked: 47067
Available plays: 91058
Play-count mismatches: 0


Across 47,067 participants, the function returned 91,058 play matrices, with no count mismatches against `nPlays`.

### Matrix shape and trial numbering

Check that every available play is a 30 × 15 trial matrix and contains trial numbers 1–30 in order. Record exceptions by participant ID and available-play index.

In [8]:
shape_issues = []
trial_number_issues = []
expected_trials = np.arange(1, 31)

for participant in participants:
    for play_index, play in enumerate(get_plays(participant), start=1):
        if play.shape != (30, 15):
            shape_issues.append({
                "id": participant["id"],
                "available_play": play_index,
                "shape": play.shape,
            })
            continue  # inspect unexpected shapes separately

        if not np.array_equal(play[:, 0], expected_trials):
            trial_number_issues.append({
                "id": participant["id"],
                "available_play": play_index,
                "trial_numbers": play[:, 0].copy(),
            })

print("Unexpected shapes:", len(shape_issues))
print("Unexpected trial numbering:", len(trial_number_issues))

Unexpected shapes: 0
Unexpected trial numbering: 0


All available play matrices have the expected dimensions and sequential trial numbering. This establishes structural consistency but doesn't yet establish that the values within each trial are complete or valid.

## 3. Happiness ratings

### Where ratings occur

Count the recorded rating schedules before applying the baseline correction described in notebook 1.

In [ ]:
from collections import Counter

rating_schedules = Counter()

for participant in participants:
    for play in get_plays(participant):
        # return trial id from trials with ratings (column 9)
        rating_trials = play[~np.isnan(play[:, 9]), 0].astype(int) 
        rating_schedules[tuple(rating_trials)] += 1

for schedule, count in rating_schedules.most_common():
    print("Stored rating rows:", schedule, "| Plays:", count)

Stored rating rows: (1, 3, 6, 9, 11, 14, 17, 20, 22, 25, 28, 30) | Plays: 91058


All 91,058 available plays share the same recorded happiness-rating schedule: a baseline rating stored in row 1, followed by ratings after trials 3, 6, 9, 11, 14, 17, 20, 22, 25, 28 and 30.

### Rating values and constant plays

Check whether recorded happiness values are finite and within 0–100, and identify plays with identical ratings throughout. Record these findings without excluding plays.

In [11]:
invalid_happiness = []
constant_happiness = []

for participant in participants:
    for play_index, play in enumerate(get_plays(participant), start=1):
        values = play[:, 9]
        ratings = values[~np.isnan(values)]

        if np.any(~np.isfinite(ratings) | (ratings < 0) | (ratings > 100)):
            invalid_happiness.append({
                "id": participant["id"],
                "available_play": play_index,
                "ratings": ratings.copy(),
            })

        if len(ratings) > 0 and np.all(ratings == ratings[0]):
            constant_happiness.append({
                "id": participant["id"],
                "available_play": play_index,
                "happiness": ratings[0],
            })

print("Plays with invalid happiness values:", len(invalid_happiness))
print("Plays with constant happiness:", len(constant_happiness))

Plays with invalid happiness values: 0
Plays with constant happiness: 4442


No recorded happiness values fall outside the permitted range or are infinite. In 4,442 plays (4.9%), all 12 ratings are identical. These plays are flagged for later consideration; no exclusions have been applied.

In [12]:
# check which happiness values are repeated
constant_values = Counter(
    record["happiness"] for record in constant_happiness
)

pd.DataFrame(
    constant_values.most_common(),
    columns=["repeated_happiness_value", "plays"],
)

,repeated_happiness_value,plays
0,50.0,3036
1,100.0,1172
2,0.0,234


Among plays with constant happiness ratings, 3,036 repeat 50, 1,172 repeat 100 and 234 repeat 0. These patterns describe the recorded responses but do not establish why participants gave constant ratings.

**Follow-up:** check slider starting values (`startValue`) by task version before interpreting the pile-up at 50.

## 4. Reward reconstruction inputs

Check choice codes, finite reward values and whether each outcome matches the option chosen.

In [13]:
invalid_gamble_choices = []

for participant in participants:
    for play_index, play in enumerate(get_plays(participant), start=1):
        gamble_choices = play[:, 6]
        invalid = ~np.isin(gamble_choices, [0, 1])

        if np.any(invalid):
            invalid_gamble_choices.append({
                "id": participant["id"],
                "available_play": play_index,
                "trial_numbers": play[invalid, 0].copy(),
                "values": gamble_choices[invalid].copy(),
            })

print("Plays with invalid choice codes:", len(invalid_gamble_choices))

Plays with invalid choice codes: 0


No invalid choice codes were found across the available plays.

### Finite reward values

Negative and zero rewards are allowed; check for missing or infinite values.

In [14]:
invalid_reward_magnitudes = []

reward_columns = [2, 3, 4, 7]
reward_names = ["certainValue", "winValue", "loseValue", "outcome"]

for participant in participants:
    for play_index, play in enumerate(get_plays(participant), start=1):
        reward_values = play[:, reward_columns]
        invalid = ~np.isfinite(reward_values)

        if np.any(invalid):
            rows, columns = np.where(invalid)

            invalid_reward_magnitudes.append({
                "id": participant["id"],
                "available_play": play_index,
                "trial_numbers": play[rows, 0].copy(),
                "fields": [reward_names[column] for column in columns],
                "values": reward_values[rows, columns].copy(),
            })

print("Plays with non-finite reward values:", len(invalid_reward_magnitudes))

Plays with non-finite reward values: 0


Certain amounts, gamble amounts and actual outcomes are finite in every available trial.

### Outcomes match the chosen option

In [15]:
invalid_outcomes = []

for participant in participants:
    for play_index, play in enumerate(get_plays(participant), start=1):
        chose_gamble = play[:, 6] == 1
        outcome = play[:, 7]

        matches_certain = outcome == play[:, 2]
        matches_gamble = (
            (outcome == play[:, 3]) | (outcome == play[:, 4])
        )

        valid = np.where(chose_gamble, matches_gamble, matches_certain)

        if np.any(~valid):
            invalid_outcomes.append({
                "id": participant["id"],
                "available_play": play_index,
                "trials": play[~valid][:, [0, 2, 3, 4, 6, 7]].copy(),
            })

print("Plays with outcomes inconsistent with choice:", len(invalid_outcomes))

Plays with outcomes inconsistent with choice: 0


Across all available trials, certain choices produced the stated certain amount, and gamble choices produced one of the two stated gamble outcomes. No inconsistencies were found. Together with valid choice codes and finite reward values, this supports reconstructing the gamble EV and RPE terms using the documented 50/50 probabilities.

These checks validate the recorded inputs and outcome relationships; they do not independently test the task probabilities.

## 5. Play metadata

### One metadata entry per available play

In [20]:
metadata_fields = [
    "timesPlayed", "dayNumber", "timeOfDay", "designVersion"
]
metadata_issues = []

for participant in participants:
    number_of_plays = len(get_plays(participant))

    for field in metadata_fields:
        values = np.atleast_1d(participant[field]) # convert single value to one-element array while leaving existing arrays untouched

        if len(values) != number_of_plays:
            metadata_issues.append({
                "id": participant["id"],
                "field": field,
                "metadata_entries": len(values),
                "available_plays": number_of_plays,
            })

print("Metadata count mismatches:", len(metadata_issues))

Metadata count mismatches: 0


`timesPlayed`, `dayNumber`, `timeOfDay` and `designVersion` each have the expected number of entries for every participant. This supports pairing metadata with play matrices by position, although matching counts alone does not verify that the pairing is correct.

### Task versions

Count plays by version, then check whether people have plays from more than one version.

In [21]:
version_counts = Counter()

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])
    version_counts.update(versions.tolist())

pd.DataFrame(
    sorted(version_counts.items()),
    columns=["designVersion", "available_plays"],
)

,designVersion,available_plays
0,1,35214
1,2,1178
2,3,5456
3,4,10684
4,5,38526


The dataset contains five task versions. Versions 1 and 5 contribute the most available plays. These counts describe plays rather than unique participants; a participant may contribute multiple plays.

In [22]:
multiple_version_participants = []

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])
    unique_versions = np.unique(versions)

    if len(unique_versions) > 1:
        multiple_version_participants.append({
            "id": participant["id"],
            "versions": unique_versions.tolist(),
            "available_plays": len(get_plays(participant)),
        })

print(
    "Participants with multiple task versions:",
    len(multiple_version_participants),
)

Participants with multiple task versions: 1132


For 1,132 participants, available plays span multiple task versions. Task version cannot be represented by a single value per participant and must be retained for each play.

### Available plays versus attempts started

The first stored play is not necessarily the first attempt. Check how often these differ.

In [23]:
missing_first_attempt = []

for participant in participants:
    attempts = np.atleast_1d(participant["timesPlayed"])

    if attempts[0] != 1:
        missing_first_attempt.append({
            "id": participant["id"],
            "first_available_attempt": attempts[0],
        })

print(
    "Participants whose first available play is not attempt 1:",
    len(missing_first_attempt),
)

Participants whose first available play is not attempt 1: 4687


For 4,687 participants, the first stored play is **not** recorded as attempt 1. Selecting the first available play per participant would therefore not necessarily select their first-ever attempt.

### Attempt counter anomalies

Look for repeated or decreasing counters in the stored sequence. Gaps are allowed.

In [25]:
attempt_order_issues = []

for participant in participants:
    attempts = np.atleast_1d(participant["timesPlayed"]).astype(np.int64) # use signed integers - unsigned differences can wrap a decrease into a positive value

    if np.any(np.diff(attempts) <= 0): # difference between consecutive attempts is non-positive
        attempt_order_issues.append({
            "id": participant["id"],
            "timesPlayed": attempts.copy(),
        })

print("Participants with non-increasing attempt counters:",
      len(attempt_order_issues))

for issue in attempt_order_issues:
    print(issue)

Participants with non-increasing attempt counters: 4
{'id': 2717, 'timesPlayed': array([ 1,  2,  3,  4,  5,  7,  9, 10,  3,  5,  8,  9, 10, 11, 12, 13, 14,
       16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 30, 32, 34, 35])}
{'id': 25501, 'timesPlayed': array([4, 5, 5])}
{'id': 30010, 'timesPlayed': array([2, 3, 2])}
{'id': 44439, 'timesPlayed': array([1, 1, 2, 3])}


Four participants have non-increasing attempt counters in the stored play sequence. IDs 25501 and 44439 have repeated adjacent counters; IDs 2717 and 30010 have decreases. These records are flagged for inspection. Stored play order is preserved, and no plays have been excluded.

Compare the trial matrices for the flagged adjacent pairs. Matching matrices warrant a closer look, rather than automatic deletion.

In [27]:
flagged_ids = {issue["id"] for issue in attempt_order_issues}

for participant in participants:
    if participant["id"] not in flagged_ids:
        continue

    attempts = np.atleast_1d(
        participant["timesPlayed"]
    ).astype(np.int64)
    plays = get_plays(participant)

    for index in np.flatnonzero(np.diff(attempts) <= 0): # indices where the diff b/w consecutive attempts is non-positive
        # then look for identical consecutive matrices
        identical = np.array_equal(
            plays[index], plays[index + 1], equal_nan=True # treat missing values as equal
        )

        print(
            "ID:", participant["id"],
            "| Available plays:", index + 1, "and", index + 2,
            "| Attempt counters:", attempts[index:index + 2],
            "| Identical trial matrices:", identical,
        )

ID: 2717 | Available plays: 8 and 9 | Attempt counters: [10  3] | Identical trial matrices: False
ID: 25501 | Available plays: 2 and 3 | Attempt counters: [5 5] | Identical trial matrices: True
ID: 30010 | Available plays: 2 and 3 | Attempt counters: [3 2] | Identical trial matrices: False
ID: 44439 | Available plays: 1 and 2 | Attempt counters: [1 1] | Identical trial matrices: True


Both flagged pairs with *repeated* attempt counters have identical trial matrices, including matching missing entries. The two pairs with *decreasing* counters have different matrices. Their causes remain unresolved; no sorting, deduplication or exclusions have been applied.

## 6. Slider starting values

Inspect recorded slider starting values (`startValue`) by task version,
using only rows with happiness ratings. This may help interpret constant
ratings.

In [29]:
starting_values_by_version = {}

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        version = int(versions[play_index])
        has_rating = ~np.isnan(play[:, 9])
        starts = play[has_rating, 10] # col 10 = "startValue"

        if version not in starting_values_by_version:
            starting_values_by_version[version] = []

        starting_values_by_version[version].extend(starts.tolist())

In [30]:
for version, values in sorted(starting_values_by_version.items()):
    counts = Counter(values)
    print(
        "Version:", version,
        "| Rating rows:", len(values),
        "| Distinct starting values:", len(counts),
        "| Most common:", counts.most_common(5),
    )

Version: 1 | Rating rows: 422568 | Distinct starting values: 38 | Most common: [(50.0, 422484), (62.0, 5), (45.0, 5), (56.00000000000001, 5), (33.0, 5)]
Version: 2 | Rating rows: 14136 | Distinct starting values: 1 | Most common: [(50.0, 14136)]
Version: 3 | Rating rows: 65472 | Distinct starting values: 50 | Most common: [(71.0, 1391), (72.0, 1365), (54.0, 1363), (65.0, 1356), (48.0, 1353)]
Version: 4 | Rating rows: 128208 | Distinct starting values: 50 | Most common: [(40.0, 2644), (56.00000000000001, 2628), (55.00000000000001, 2624), (59.0, 2614), (42.0, 2609)]
Version: 5 | Rating rows: 462312 | Distinct starting values: 50 | Most common: [(66.0, 9502), (59.0, 9429), (43.0, 9383), (55.00000000000001, 9382), (33.0, 9370)]


In [31]:
start_summary = []

for version, values in sorted(starting_values_by_version.items()):
    values = np.asarray(values)
    finite = values[np.isfinite(values)]

    start_summary.append({
        "designVersion": version,
        "rating_rows": len(values),
        "non_finite": np.sum(~np.isfinite(values)),
        "minimum": finite.min() if len(finite) else np.nan,
        "maximum": finite.max() if len(finite) else np.nan,
    })

pd.DataFrame(start_summary)

,designVersion,rating_rows,non_finite,minimum,maximum
0,1,422568,0,25.0,74.0
1,2,14136,0,50.0,50.0
2,3,65472,0,25.0,74.0
3,4,128208,0,25.0,74.0
4,5,462312,0,25.0,74.0


Slider starting values differ across task versions. Version 2 always starts at 50, while versions 3–5 contain 50 distinct starting values spanning 25–74. Version 1 is mostly fixed at 50 with only 84 rating rows showing other starting values. No missing or infinite starting values were found.

In [32]:
version1_start_exceptions = []

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        if versions[play_index] != 1:
            continue

        has_rating = ~np.isnan(play[:, 9])
        unusual = has_rating & (play[:, 10] != 50)

        if np.any(unusual):
            version1_start_exceptions.append({
                "id": participant["id"],
                "available_play": play_index + 1,
                "rating_rows_not_starting_at_50": int(unusual.sum()),
            })

pd.DataFrame(version1_start_exceptions)

,id,available_play,rating_rows_not_starting_at_50
0,16258,1,12
1,25501,1,12
2,25501,2,12
3,25501,3,12
4,25678,1,12
5,25678,2,12
6,39085,1,12


The version-1 exceptions occur in seven plays from four participants, with all 12 rating rows affected in each play. 

In [34]:
constant_plays_by_version = Counter()

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        ratings = play[~np.isnan(play[:, 9]), 9]

        if len(ratings) > 0 and np.all(ratings == ratings[0]):
            version = int(versions[play_index])
            constant_plays_by_version[version] += 1

constant_summary = pd.DataFrame([
    {
        "designVersion": version,
        "available_plays": total,
        "constant_plays": constant_plays_by_version[version],
        "constant_percent": 100 * constant_plays_by_version[version] / total,
    }
    for version, total in sorted(version_counts.items())
])

constant_summary.style.format({
    "constant_percent": "{:.2f}",
}).hide(axis="index")

designVersion,available_plays,constant_plays,constant_percent
1,35214,3474,9.87
2,1178,224,19.02
3,5456,81,1.48
4,10684,133,1.24
5,38526,530,1.38


The proportion of plays with constant happiness ratings differs across task versions. Versions 1–2 (whose recorded slider starting values are predominantly or entirely 50) have higher proportions than versions 3–5, whose starting values vary. This comparison doesn't establish that starting position caused constant responding: versions also differ in other respects, and participants can contribute multiple plays.

### Constant midpoint and endpoint ratings by version
Separate constant midpoint ratings (50) from constant endpoint ratings (0 and 100) within each task version. Percentages use all available plays in that version as the denominator.

In [35]:
constant_values_by_version = Counter()

for participant in participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        ratings = play[~np.isnan(play[:, 9]), 9]

        if len(ratings) > 0 and np.all(ratings == ratings[0]):
            version = int(versions[play_index])
            constant_values_by_version[(version, ratings[0])] += 1

constant_value_summary = pd.DataFrame([
    {
        "designVersion": version,
        "constant_value": value,
        "plays": constant_values_by_version[(version, value)],
        "percent_of_version_plays": (
            100 * constant_values_by_version[(version, value)]
            / version_counts[version]
        ),
    }
    for version in sorted(version_counts)
    for value in [0, 50, 100]
])

constant_value_summary.style.format({
    "constant_value": "{:.0f}",
    "percent_of_version_plays": "{:.2f}",
}).hide(axis="index")

designVersion,constant_value,plays,percent_of_version_plays
1,0,62,0.18
1,50,2829,8.03
1,100,583,1.66
2,0,2,0.17
2,50,207,17.57
2,100,15,1.27
3,0,17,0.31
3,50,0,0.00
3,100,64,1.17
4,0,25,0.23


Constant midpoint ratings occur only in versions 1–2: 8.03% of version-1 plays and 17.57% of version-2 plays have all ratings equal to 50. In versions 3–5, constant ratings occur only at the endpoints, 0 or 100. The version difference in overall constant responding is therefore mainly concentrated at the midpoint. This pattern alone does not establish whether participants left the slider untouched or deliberately selected these values.

## 7. Notes and next steps

The trial structure, rating schedule and reward fields pass the checks above. Constant happiness ratings and attempt-counter anomalies are recorded separately; nothing has been removed.

Keeping participant ID and the available-play index as the stored-play identifier, with `timesPlayed` and task version alongside them. Still to look at: slider starting values and task-version differences, and the metadata attached to the flagged identical matrices. Decisions about exclusions and how to represent reward exposure between ratings remain open.